# CombBandits — GPU experiments

Runs two experiments on Colab A100:
- **exp8_scaling_d** — GPU-batched dimension scaling (no LLM, pure bandit simulation)
- **exp9_local** — Real Llama 4 Scout 17B oracle with **full weight tracking**

**Setup:** Runtime → Change runtime type → **A100 GPU** (Colab Premium)

**Weight tracking captures per oracle call:**
- Token log-probabilities for each suggested arm (model confidence)
- Attention weights over arm metadata (what the model reads)
- Hidden state vectors (128-d PCA) — tracks representation drift over rounds
- Output entropy — model uncertainty
- Cross-query KL divergence — internal vs output consistency

In [ ]:
# ── 0. Verify GPU ─────────────────────────────────────────────────────────────
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Switch to GPU runtime first (Runtime > Change runtime type)'
print(f'CUDA device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
!git clone https://github.com/vkmk1/CombBandits /content/CombBandits
%cd /content/CombBandits

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q -e '.[dev]'
!pip install -q transformers>=4.47 accelerate bitsandbytes huggingface_hub
print('Install complete.')

In [ ]:
# ── 3. HuggingFace token (needed for Llama 4 gated model) ────────────────────
# Get a free token at https://huggingface.co/settings/tokens
# Then accept the Llama 4 license at https://huggingface.co/meta-llama/Llama-4-Scout-17B-16E-Instruct
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')  # set in Colab Secrets panel (key icon)
os.environ['HF_TOKEN'] = HF_TOKEN

# patch exp9_local.yaml to inject the token
import yaml
with open('configs/experiments/exp9_local.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['oracles'][0]['hf_token'] = HF_TOKEN
with open('configs/experiments/exp9_local.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('HF token injected into config.')

In [ ]:
# ── 4. exp8_scaling_d — GPU batched bandit simulation ─────────────────────────
import time
print('Running exp8_scaling_d (GPU batched, ~15 min)...')
t0 = time.time()
!python -m combbandits.cli run-gpu configs/experiments/exp8_scaling_d.yaml \
    --output-dir results/exp8_scaling_d --device cuda
print(f'exp8 done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── 5. exp9_local — Llama 4 Scout with weight tracking ────────────────────────
# workers=1: model is loaded once and reused (not reloaded per worker)
print('Running exp9_local (Llama 4 Scout, weight tracking, ~90 min)...')
t0 = time.time()
!python -m combbandits.cli run configs/experiments/exp9_local.yaml \
    --output-dir results/exp9_local --workers 1
print(f'exp9_local done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── 6. Generate figures ───────────────────────────────────────────────────────
for exp in ['exp8_scaling_d', 'exp9_local']:
    !python -m combbandits.cli plot results/{exp}/{exp}_results.json --output-dir figures/{exp}
    !python -m combbandits.cli metrics results/{exp}/{exp}_results.json
    print(f'{exp}: figures saved')

In [ ]:
# ── 7. Weight tracking analysis ───────────────────────────────────────────────
import sqlite3, json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

db = sqlite3.connect('metadata/oracle_weights.db')

calls = pd.read_sql('SELECT * FROM oracle_calls ORDER BY call_id', db)
groups = pd.read_sql('SELECT * FROM query_groups ORDER BY group_id', db)
db.close()

print(f'Total oracle calls tracked: {len(calls)}')
print(f'Total query groups (rounds): {len(groups)}')
print()
print(calls[['trial_round','query_variant','suggestion_logprob',
             'output_entropy','attn_on_metadata','hidden_state_norm']].describe().round(3))

In [ ]:
# ── 8. Plot: entropy + attention + logprob over bandit rounds ─────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Primary queries only (variant=0)
primary = calls[calls['query_variant'] == 0].copy()

axes[0,0].plot(primary['trial_round'], primary['output_entropy'], alpha=0.6, color='steelblue')
axes[0,0].set_title('Output entropy over rounds')
axes[0,0].set_xlabel('Bandit round'); axes[0,0].set_ylabel('Entropy')

axes[0,1].plot(primary['trial_round'], primary['attn_on_metadata'], alpha=0.6, color='crimson')
axes[0,1].set_title('Attention on arm metadata over rounds')
axes[0,1].set_xlabel('Bandit round'); axes[0,1].set_ylabel('Mean attention weight')

axes[1,0].plot(primary['trial_round'], primary['suggestion_logprob'], alpha=0.6, color='green')
axes[1,0].set_title('Log-prob of suggestion over rounds')
axes[1,0].set_xlabel('Bandit round'); axes[1,0].set_ylabel('Σ log p(token)')

axes[1,1].plot(groups['trial_round'], groups['kappa'], label='κ (output overlap)', color='orange')
axes[1,1].plot(groups['trial_round'], groups['hidden_kl_div'], label='hidden KL div proxy', color='purple')
axes[1,1].set_title('Output kappa vs internal hidden diversity')
axes[1,1].set_xlabel('Bandit round'); axes[1,1].legend()

plt.suptitle('Llama 4 Scout Oracle — Internal State Tracking', fontsize=13)
plt.tight_layout()
plt.savefig('figures/exp9_local/weight_tracking.png', dpi=150)
plt.show()
print('Saved: figures/exp9_local/weight_tracking.png')

In [ ]:
# ── 9. Plot: hidden-state PCA trajectory (2D) ────────────────────────────────
from sklearn.decomposition import PCA

primary = calls[calls['query_variant'] == 0].copy()
hidden_vecs = np.array([json.loads(r) for r in primary['hidden_state_pca']])

pca = PCA(n_components=2)
coords = pca.fit_transform(hidden_vecs)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(coords[:, 0], coords[:, 1],
                c=primary['trial_round'].values,
                cmap='viridis', alpha=0.7, s=20)
plt.colorbar(sc, label='Bandit round')
ax.set_title('Hidden state PCA trajectory — Llama 4 Scout oracle\n(colour = bandit round; drift shows representation change over learning)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.tight_layout()
plt.savefig('figures/exp9_local/hidden_state_pca.png', dpi=150)
plt.show()
print('Saved: figures/exp9_local/hidden_state_pca.png')

In [ ]:
# ── 10. Upload to S3 ─────────────────────────────────────────────────────────
import os, boto3

AWS_ACCESS_KEY_ID     = ''  # paste or use Colab Secrets
AWS_SECRET_ACCESS_KEY = ''
S3_BUCKET = 'combbandits-results-099841456154'

if not AWS_ACCESS_KEY_ID:
    try:
        from google.colab import userdata
        AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
        AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
    except Exception:
        print('No AWS creds — skipping S3 upload. Download from Files panel manually.')

if AWS_ACCESS_KEY_ID:
    os.environ['AWS_ACCESS_KEY_ID']     = AWS_ACCESS_KEY_ID
    os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
    os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
    s3 = boto3.client('s3')
    for local, prefix in [('results/exp8_scaling_d','results/exp8_scaling_d'),
                           ('results/exp9_local','results/exp9_local'),
                           ('figures/exp8_scaling_d','figures/exp8_scaling_d'),
                           ('figures/exp9_local','figures/exp9_local'),
                           ('metadata','metadata')]:
        for root, _, files in os.walk(local):
            for f in files:
                full = os.path.join(root, f)
                key  = full.replace(local, prefix, 1)
                s3.upload_file(full, S3_BUCKET, key)
                print(f'  ↑ s3://{S3_BUCKET}/{key}')
    print('Upload complete.')